In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path
from tqdm.auto import tqdm
from pd_estim_A.data.data_import import (
    load_data, load_ecb_1y_yield,
    fill_liabilities, drop_high_leverage_firms,
    prepare_nig_inputs
)
from pd_estim_A.data.cds_df import get_cds_panel
from pd_estim_A.models.nig.nig_apath import (
    NIGParams,
    invert_assets_weekly_for_firm,
)

from pd_estim_A.models.nig.nig_em import (
    fit_nig_params_from_weekly_assets,
)

from pd_estim_A.models.nig.nig_pd import (
    pd_terminal_nig_weekly,
    pd_weekly_one_firm,
)

In [ ]:
# Paths
print(Path.cwd())
data_path = Path.cwd() / ".." / "data" / "raw"
output_path = Path.cwd() / ".." / "data" / "derived"

# Load data and prepare panels
ret_daily, bs, coverage = load_data(
    data_path / "Jan2025_Accenture_Dataset_ErasmusCase.xlsx",
    start_date="2012-01-01",
    end_date="2025-12-19",
    enforce_coverage=True,
    coverage_tol=0.995,
    liabilities_scale="auto",
    verbose=True,
)

df_rf = load_ecb_1y_yield(
    startPeriod="2010-01-01",
    endPeriod="2025-12-31",
    out_file= output_path / "ecb_yc_1y_aaa.xml",
    verify_ssl=True,  # recommended if it works
)

df_cal = ret_daily[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)

debt_daily = fill_liabilities(bs, df_cal)

ret_filt, bs_filt, lev_by_firm, dropped = drop_high_leverage_firms(
    ret_daily,
    bs,
    df_calendar=df_cal,
    debt_daily=debt_daily,
    lev_threshold=8.0,
    lev_agg="median",
    verbose=True,
)

# keep debt panel consistent with filtered firms
keep = set(ret_filt["gvkey"].astype(str).unique())
debt_daily_filt = debt_daily[debt_daily["gvkey"].astype(str).isin(keep)].copy()


nig_df, em_cache = prepare_nig_inputs(ret_filt, bs_filt, df_rf, debt_daily=debt_daily_filt, build_em=False)
print(nig_df.head())
print(nig_df.shape)
print(nig_df.describe())

In [ ]:
# call cds panel and merge
cds = get_cds_panel(
    project_root= Path.cwd() / "..",
    save_csv=False,
    verbose=True,
)
# ensure types
merton = nig_df.copy()
merton["gvkey"] = merton["gvkey"].astype(str)
merton["date"]  = pd.to_datetime(merton["date"])

cds["gvkey"] = cds["gvkey"].astype(str)
cds["date"]  = pd.to_datetime(cds["date"])

# keep only firms that exist in BOTH (drop firms with no CDS)
common_gv = sorted(set(merton["gvkey"].unique()) & set(cds["gvkey"].unique()))
merton = merton[merton["gvkey"].isin(common_gv)].copy()
cds = cds[cds["gvkey"].isin(common_gv)].copy()

# also drop CDS rows whose dates are outside merged's date range
dmin, dmax = merton["date"].min(), merton["date"].max()
cds = cds[(cds["date"] >= dmin) & (cds["date"] <= dmax)].copy()

# merge-asof onto merged's dates (direction='backward')
merton = merton.sort_values(["date", "gvkey"]).reset_index(drop=True)
cds    = cds.sort_values(["date", "gvkey"]).reset_index(drop=True)

merged_cds = pd.merge_asof(
    merton,
    cds,
    on="date",
    by="gvkey",
    direction="backward",
    allow_exact_matches=True,
)

# drop rows where CDS still missing
nig_df = merged_cds.dropna(subset=["cds"]).reset_index(drop=True)

print("firms after intersection:", nig_df["gvkey"].nunique())
print("rows after merge:", len(nig_df))
print("date range:", nig_df["date"].min(), "→", nig_df["date"].max())

In [ ]:
# Cell 2 — rolling configuration

TRAIN_YEARS = 2
STEP_FREQ = "QE"
WEEK_ENDING = "W-FRI"

T_INV = 1.0                  # 1Y maturity used in inversion
HORIZON_WEEKS = 52.0         # 1Y PD horizon on weekly scale

DATA_END = pd.Timestamp("2024-12-31")   # keep equal to Merton if you want clean comparison
LAST_TRAIN_END = DATA_END - pd.offsets.QuarterEnd(1)

MIN_DAILY_ROWS = 10
MIN_WEEKLY_RETURNS = 60      # require enough weekly implied asset returns in each 2Y window

P0 = NIGParams(alpha=15.0, beta=-3.0, delta=0.20, mu=0.00)

INVERT_U = 120.0
INVERT_N = 2000

EM_MAX_ITER = 80
EM_TOL = 1e-6

MAX_FIRMS = None
MAX_WINDOWS = None

In [ ]:
# Cell 3 — panel preparation
# Assumes nig_df already exists and has at least:
# gvkey, date, E, L, r
# plus optionally company, country_iso, etc.

panel = nig_df.copy()
panel["gvkey"] = panel["gvkey"].astype(str)
panel["date"] = pd.to_datetime(panel["date"])

needed_cols = ["gvkey", "date", "company", "E", "L", "r"]
panel = panel[[c for c in needed_cols if c in panel.columns]].copy()

for c in ["E", "L", "r"]:
    panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel = (
    panel.dropna(subset=["gvkey", "date", "E", "L", "r"])
         .query("E > 0 and L > 0")
         .sort_values(["gvkey", "date"])
         .reset_index(drop=True)
)

firm_daily = {}
for gvkey, g in panel.groupby("gvkey", sort=False):
    g = g.sort_values("date").groupby("date", as_index=False).last()
    firm_daily[gvkey] = g.set_index("date")

gvkeys_all = sorted(firm_daily.keys())
if MAX_FIRMS is not None:
    gvkeys_all = gvkeys_all[:int(MAX_FIRMS)]

print("Firms loaded:", len(firm_daily), "| Firms in run:", len(gvkeys_all))
print("Panel date range:", panel["date"].min().date(), "to", panel["date"].max().date())
print("LAST_TRAIN_END:", LAST_TRAIN_END.date(), "| DATA_END:", DATA_END.date())
display(panel.head())

In [ ]:
# Cell 4 — build the rolling window schedule

global_min_date = panel["date"].min()
earliest_end = global_min_date + pd.DateOffset(years=TRAIN_YEARS) - pd.Timedelta(days=1)

train_ends = pd.date_range(start=earliest_end, end=LAST_TRAIN_END, freq=STEP_FREQ)
train_ends = pd.to_datetime(train_ends)

if MAX_WINDOWS is not None:
    train_ends = train_ends[:int(MAX_WINDOWS)]

windows = []
for train_end in train_ends:
    train_start = train_end - pd.DateOffset(years=TRAIN_YEARS) + pd.Timedelta(days=1)
    oos_start = train_end + pd.Timedelta(days=1)
    oos_end = train_end + pd.offsets.QuarterEnd(1)

    windows.append(
        {
            "train_start": pd.Timestamp(train_start),
            "train_end": pd.Timestamp(train_end),
            "oos_start": pd.Timestamp(oos_start),
            "oos_end": pd.Timestamp(oos_end),
        }
    )

windows_df = pd.DataFrame(windows)
display(windows_df.head())
display(windows_df.tail())
print("n_windows:", len(windows_df))